# Auto Video Producer — Pipeline no Google Colab (GPU T4)

Este notebook instala e executa o pipeline **Auto Video Producer** dentro do Google Colab, usando a GPU **T4** para acelerar:
- a **geração de áudio via OmniVoice** (TTS local com voice design);
- a **classificação temática por IA** do vídeo de fundo (OpenAI-compatible API).

**Antes de começar:** no menu do Colab, vá em `Ambiente de execução` → `Alterar o tipo de ambiente de execução` → GPU → **T4**.

**Importante sobre a interface gráfica:** `gui.py` usa Tkinter, que precisa de uma tela (display) e **não funciona dentro do Colab**. Este notebook substitui a GUI por células de código equivalentes (upload de arquivos, botões via `ipywidgets`, etc.). Para usar a GUI de verdade, rode o projeto localmente na sua máquina (veja o `README.md`).

**Persistência:** o Colab apaga `/content` toda vez que o ambiente de execução reinicia. Este notebook monta o **Google Drive** e instala o projeto lá, para que seus assets (vídeos, músicas), o banco de dados e os vídeos gerados sobrevivam entre sessões. As instalações de sistema (ffmpeg, pacotes Python) precisam ser refeitas a cada nova sessão — é normal, é assim que o Colab funciona.

## 1. Verificar a GPU disponível

In [1]:
!nvidia-smi

## 2. Montar o Google Drive

Recomendado para manter seus assets, o banco de dados (`videos.db`) e os vídeos gerados salvos entre sessões. Se preferir não usar o Drive (tudo será perdido ao reiniciar o ambiente), pule esta célula e ajuste `PROJECT_ROOT` para `/content/auto-video-producer`.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Definir caminhos e instalar dependências do sistema

In [3]:
import os
PROJECT_ROOT = "/content/drive/MyDrive/auto-video-producer"
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)
print(f"Projeto será instalado em: {PROJECT_ROOT}")

# Instala ffmpeg e dependências do sistema
!apt-get update -qq && apt-get install -y -qq ffmpeg 2>/dev/null
print("ffmpeg instalado:", os.system("ffmpeg -version | head -1"))

## 4. Enviar código do projeto

Envie o arquivo **`auto-video-producer.zip`** (o pacote de código do projeto, sem assets pesados) quando solicitado abaixo. O notebook vai extrair automaticamente.

In [4]:
import zipfile, shutil, os
from google.colab import files

uploaded = files.upload()

for fname, content in uploaded.items():
    if fname.endswith('.zip'):
        zip_path = os.path.join(PROJECT_ROOT, fname)
        with open(zip_path, 'wb') as f:
            f.write(content)
        print(f"Extraindo {fname}...")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(PROJECT_ROOT)
        os.remove(zip_path)
        print("Extração concluída.")
        break
else:
    print("Nenhum .zip enviado. Se já extraiu antes, ignore.")

## 5. Instalar dependências Python

In [5]:
!pip install -q -r requirements.txt
print("Dependências instaladas.")

## 6. Criar estrutura de pastas de assets (se não existirem)

In [6]:
import os
assets_dirs = [
    "assets/videos/natureza",
    "assets/videos/cidade",
    "assets/videos/tecnologia",
    "assets/videos/pessoas",
    "assets/videos/agua",
    "assets/videos/ceu",
    "assets/videos/abstrato",
    "assets/videos/escritorio",
    "assets/videos/casa",
    "assets/musicas",
    "assets/fontes",
    "assets/overlays",
    "assets/logo",
    "input/textos",
    "input/audio",
    "input/thumbs",
    "cache",
    "output/videos",
    "output/shorts",
    "database",
    "logs"
]
for d in assets_dirs:
    os.makedirs(os.path.join(PROJECT_ROOT, d), exist_ok=True)
print("Estrutura de pastas criada.")

## 7. Configurar `config.yaml`

Copie `config.yaml.example` para `config.yaml` e edite com suas chaves de API.

In [7]:
import shutil
src = os.path.join(PROJECT_ROOT, "config.yaml.example")
dst = os.path.join(PROJECT_ROOT, "config.yaml")
if not os.path.exists(dst):
    shutil.copy2(src, dst)
    print("config.yaml criado a partir do template. EDITE AGORA!")
else:
    print("config.yaml já existe.")

# Mostra o conteúdo para você editar
with open(dst, 'r') as f:
    print(f.read())

## 8. Editar `config.yaml` com suas chaves

Edite o arquivo acima (dois cliques no arquivo no painel de arquivos à esquerda) e preencha:
- `classification.endpoint`, `api_key`, `model`
- `llm.endpoint`, `api_key`, `model`
- `tts.provider` e configurações
- `youtube.client_secrets_file`, `token_file`

Depois de editar, salve (Ctrl+S) e rode a próxima célula.

## 9. Popular assets de vídeo/música

Adicione seus vídeos de fundo em `assets/videos/<categoria>/` e músicas em `assets/musicas/`.
Use o notebook `baixar_assets.ipynb` para baixar de bancos de stock gratuitos (Pexels, Pixabay, etc.) direto pro Drive.

## 10. Indexar assets no banco de dados

In [8]:
!python main.py scan

## 11. Gerar vídeo a partir de texto (TTS -> vídeo)

Crie um arquivo `.txt` em `input/textos/` ou faça upload aqui.

In [9]:
# Exemplo: criar um texto de teste
test_text = """Bem-vindo ao Auto Video Producer!

Este é um exemplo de geração de vídeo a partir de texto.
O sistema converte o texto em áudio usando OmniVoice,
seleciona vídeos de fundo por tema, adiciona música ambiente,
e gera o vídeo final com tela de call-to-action.

Edite este texto ou crie seus próprios arquivos em input/textos/"""

with open(os.path.join(PROJECT_ROOT, "input/textos/exemplo.txt"), "w") as f:
    f.write(test_text)
print("Arquivo de exemplo criado.")

In [10]:
# Roda pipeline completo: texto -> TTS -> vídeo
!python main.py run-text input/textos/exemplo.txt

## 12. Ver resultado

O vídeo final estará em `output/videos/` ou `output/shorts/`.

In [11]:
import glob
videos = glob.glob(os.path.join(PROJECT_ROOT, "output/**/*.mp4"), recursive=True)
if videos:
    print("Vídeos gerados:")
    for v in videos:
        size_mb = os.path.getsize(v) / (1024*1024)
        print(f"  {os.path.relpath(v, PROJECT_ROOT)} ({size_mb:.1f} MB)")
else:
    print("Nenhum vídeo gerado ainda.")

## 13. (Opcional) Upload para YouTube

Configure `youtube.client_secrets_file` e `youtube.token_file` no `config.yaml`
e use a biblioteca `google-auth-oauthlib` para autenticação.
O upload pode ser adicionado como etapa final do pipeline.